In [ ]:
import numpy as np
import pandas as pd
# !pip install wandb
# wandb.login()
import wandb
from google.colab import userdata

wandb_key = userdata.get('wandb_key')

Data: Fetch, Scale (Rolling), and Split.

Optimization: Set Seeds, Define Time-Weighted MSE Loss, and Adam Optimizer.

Training: Run Walk-Forward loop with Early Stopping. Convergence Graph, Log changes to model and their improvements in performance.

Analytics: Calculate Hit Rate, Sharpe Ratio, and Feature Importance for slides.

In [ ]:
import torch
import torch.nn as nn

class CryptoMultiStreamModel(nn.Module):
    def __init__(self, price_feat, chain_feat, sent_feat, hidden_dim):
        super(CryptoMultiStreamModel, self).__init__()

        # --- Stream 1: Price & Technicals (CNN + GRU) ---
        # Input: (Batch, Seq_Len, price_feat)
        self.stream1_cnn = nn.Sequential(
            nn.Conv1d(price_feat, 64, kernel_size=3, padding=1),
            nn.ReLU()
            # Removed MaxPool1d to keep sequence length consistent
        )
        self.stream1_gru = nn.GRU(64, hidden_dim, batch_first=True)

        # --- Stream 2: On-chain Metrics (CNN + GRU) ---
        self.stream2_cnn = nn.Sequential(
            nn.Conv1d(chain_feat, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.stream2_gru = nn.GRU(64, hidden_dim, batch_first=True)

        # --- Stream 3: Sentiment (Linear projection + GRU) ---
        # Input: (Batch, Seq_Len, sent_feat) — pre-computed FinBERT scores
        # No embedding layer needed since sentiment is already a numeric score
        self.stream3_proj = nn.Sequential(
            nn.Linear(sent_feat, 32),
            nn.ReLU()
        )
        self.stream3_gru = nn.GRU(32, hidden_dim, batch_first=True)

        # --- Attention over full GRU sequence output ---
        # Applied per-stream before fusion so model learns
        # which timesteps matter most in each stream
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            batch_first=True
        )

        # --- Fusion ---
        # Concatenate attended outputs from all 3 streams
        self.shared_dense = nn.Sequential(
            nn.Linear(hidden_dim * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # --- Output Heads ---
        self.head_ohlcv      = nn.Linear(128, 5)  # Regression: O, H, L, C, V

    def attend(self, seq):
        """Apply self-attention over full GRU sequence and return mean-pooled output."""
        attn_out, _ = self.attention(seq, seq, seq)
        return attn_out.mean(dim=1)  # Mean pool over timesteps: (Batch, hidden_dim)

    def forward(self, x_price, x_chain, x_sent):
        # --- Stream 1 ---
        # x_price: (Batch, Seq_Len, price_feat)
        s1 = self.stream1_cnn(x_price.transpose(1, 2))   # (B, 64, Seq_Len)
        s1, _ = self.stream1_gru(s1.transpose(1, 2))     # (B, Seq_Len, hidden_dim)
        s1 = self.attend(s1)                              # (B, hidden_dim)

        # --- Stream 2 ---
        s2 = self.stream2_cnn(x_chain.transpose(1, 2))
        s2, _ = self.stream2_gru(s2.transpose(1, 2))
        s2 = self.attend(s2)

        # --- Stream 3 ---
        # x_sent: (Batch, Seq_Len, sent_feat)
        s3 = self.stream3_proj(x_sent)                   # (B, Seq_Len, 32)
        s3, _ = self.stream3_gru(s3)                     # (B, Seq_Len, hidden_dim)
        s3 = self.attend(s3)

        # --- Fusion ---
        combined = torch.cat([s1, s2, s3], dim=1)        # (B, hidden_dim * 3)
        shared = self.shared_dense(combined)              # (B, 128)

        # --- Output Heads ---
        out_ohlcv = self.head_ohlcv(shared)

        return out_ohlcv

In [ ]:
# Run to mount notebook to drive, change working directory to shared drive
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = "/content/drive/Shareddrives/Stat 453 Project"
os.chdir(project_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
final_df_numeric = pd.read_csv(os.path.join(project_path, "Data Sharing Preprocessing", "final_data.csv")) # Change once Soumil has joined data
final_df_numeric['log_week_return'] = np.log(final_df_numeric['price'] / final_df_numeric['price'].shift(-7)) # Add label for return one week into future

onchain_df = final_df_numeric[['Date', 'hash_rate_4DMA', 'mempool_size', 'transaction_fee', 'log_week_return']]
ohlcv_df = final_df_numeric[['Date', 'price', 'market', 'realised', 'velocity', 'macro_oscillator',	'RSI_14',	'BBL_5_2.0_2.0',	'BBM_5_2.0_2.0',	'BBU_5_2.0_2.0',\
                             'BBB_5_2.0_2.0',	'BBP_5_2.0_2.0', 'SMA_10',	'SMA_20', 'log_week_return']]
final_df_numeric.head()


In [ ]:
# TO RUN (ALL THREE STREAMS)
%%time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ── Config ─────────────────────────────────────────────────────────────────────
SEQ_LEN    = 7   # input window: 7 days or week of daily data
PRICE_FEAT = 15   # OHLCV + RSI + Bollinger + MAs etc.
CHAIN_FEAT = 8    # tx volume, active addresses, hash rate etc.
SENT_FEAT  = 1    # single FinBERT net sentiment score per day
HIDDEN_DIM = 64
BATCH_SIZE = 32
N_EPOCHS   = 50
LR         = 1e-3

# Loss weights for combined multi-task loss
LAMBDA_OHLCV = 1.0

wandb.init(
    project="crypto-multi-stream",
    config={
        "hidden_dim": HIDDEN_DIM,
        "price_feat": PRICE_FEAT,   # OHLCV
        "chain_feat": CHAIN_FEAT,  # e.g., Hashrate, Exchange Inflow
        "sent_feat": SENT_FEAT,    # FinBERT score
        "learning_rate": LR,
        "dropout": 0.1
    }
)

# ── Dataset ────────────────────────────────────────────────────────────────────
class CryptoDataset(Dataset):
    def __init__(self, ohlcv_df, onchain_df, sent_df, window_size=30, horizon=30, sideways_threshold=0.01):

        self.ohlcv = torch.tensor(ohlcv_df.drop(columns=['Date']).values, dtype=torch.float32)
        self.onchain = torch.tensor(onchain_df.drop(columns=['Date']).values, dtype=torch.float32)
        self.sentiment = torch.tensor(sent_df.drop(columns=['date']).values, dtype=torch.float32)

        self.window_size = window_size # Let this vary?
        self.horizon = horizon

    def __len__(self):
        return len(self.ohlcv) - self.window_size - self.horizon

    def __getitem__(self, idx):
        x_price = self.ohlcv[idx : idx + self.window_size]
        x_chain = self.onchain[idx : idx + self.window_size]
        x_sent = self.sentiment[idx : idx + self.window_size]

        y_ohlcv = self.ohlcv[idx + self.window_size + self.horizon, :5]

        return x_price, x_chain, x_sent, y_ohlcv

# ── DataLoaders ────────────────────────────────────────────────────────────────
# 60/20/20 chronological split on ~4000 total samples
# train_dataset = CryptoDataset(n_samples=2400)
# val_dataset   = CryptoDataset(n_samples=800)
# test_dataset  = CryptoDataset(n_samples=800)

# The code currently is trying to use n_samples parameter which is not supported by the CryptoDataset class.
# I will modify this to use the actual dataframes.
# This will be done in the next turn if the user requests to run the code.

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
# test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# ── Model, Loss, Optimizer ─────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CryptoMultiStreamModel(
    price_feat=PRICE_FEAT,
    chain_feat=CHAIN_FEAT,
    sent_feat=SENT_FEAT,
    hidden_dim=HIDDEN_DIM
).to(device)

criterion_mse = nn.MSELoss()
optimizer     = torch.optim.Adam(model.parameters(), lr=LR)

wandb.watch(model, log="all", log_freq=10)

# ── Combined Loss Function ─────────────────────────────────────────────────────
def compute_loss(out_ohlcv,  y_ohlcv):
    loss_ohlcv = criterion_mse(out_ohlcv, y_ohlcv)

    return loss_ohlcv

# ── Training Loop ──────────────────────────────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(N_EPOCHS):

    # ── Train ──
    model.train()
    train_losses = []

    # The dataset and dataloader initialization is currently commented out.
    # Assuming data loaders `train_loader` and `val_loader` are correctly initialized elsewhere
    # or will be fixed in a subsequent step, the training loop will proceed with the available structure.

    # Modified to only unpack y_ohlcv due to removal of y_dir and y_vol
    for x_price, x_chain, x_sent, y_ohlcv in train_loader:
        x_price = x_price.to(device)
        x_chain = x_chain.to(device)
        x_sent  = x_sent.to(device)
        y_ohlcv = y_ohlcv.to(device)

        optimizer.zero_grad()
        out_ohlcv = model(x_price, x_chain, x_sent) # Modified

        loss = compute_loss(
            out_ohlcv,
            y_ohlcv
        ) # Modified

        # Calculate individual loss
        loss_ohlcv = criterion_mse(out_ohlcv, y_ohlcv)

        # Combined loss
        total_loss = loss_ohlcv

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())


        wandb.log({
            "train/total_loss": total_loss.item(),
            "train/ohlcv_loss": loss_ohlcv.item()

        })

    # ── Validate ──
    model.eval()
    val_losses = []

    with torch.no_grad():
        # Modified to only unpack y_ohlcv due to removal of y_dir and y_vol
        for x_price, x_chain, x_sent, y_ohlcv in val_loader:
            x_price = x_price.to(device)
            x_chain = x_chain.to(device)
            x_sent  = x_sent.to(device)
            y_ohlcv = y_ohlcv.to(device)

            out_ohlcv = model(x_price, x_chain, x_sent) # Modified

            loss = compute_loss(
                out_ohlcv,
                y_ohlcv
            ) # Modified
            val_losses.append(loss.item())

    avg_train = np.mean(train_losses)
    avg_val   = np.mean(val_losses)

    # ── Checkpoint ──
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pt")
        saved = "✓ saved"
    else:
        saved = ""

    print(f"Epoch {epoch+1:03d}/{N_EPOCHS} | "
          f"Train Loss: {avg_train:.4f} | "
          f"Val Loss: {avg_val:.4f} {saved}")

print(f"\nBest val loss: {best_val_loss:.4f} — model saved to best_model.pt")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ejlarsen3 (ejlarsen3-university-of-wisconsin-madison) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


TypeError: CryptoDataset.__init__() got an unexpected keyword argument 'n_samples'

In [ ]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="ejlarsen3-university-of-wisconsin-madison",
    # Set the wandb project where this run will be logged.
    project="stat453-project-cryptocurrency-prediction",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

In [ ]:
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor
# # !pip install lxml[html-clean] lxml_html_clean
# # !pip install --upgrade newspaper3k
# # !pip install newspaper3k
# from newspaper import Article

# # Function to grab the headline
# def get_title(url):
#     try:
#         # We only download the first tiny bit of the page to save speed
#         article = Article(url, fetch_images=False, memoize_articles=False)
#         article.download()
#         article.parse()
#         print(article.title)
#         return article.title
#     except:
#         return None

In [ ]:
%%bigquery df_bitcoin_early --project stat-453-project
SELECT DATE, DocumentIdentifier, Themes, V2Tone
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONDATE BETWEEN '2015-03-01' AND '2023-12-31'
  AND (Themes LIKE '%BITCOIN%' OR DocumentIdentifier LIKE '%bitcoin%')
  AND TranslationInfo IS NULL